# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaiRagab/flyrank_ml_internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose Logistic Regression because this is a binary classification task. It is simple, interpretable, and appropriate for comparing against the Week-4 baseline without adding unnecessary complexity.

In [6]:
!git clone https://github.com/MaiRagab/flyrank_ml_internship.git

Cloning into 'flyrank_ml_internship'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (105/105), done.
remote: Total 148 (delta 57), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 1.86 MiB | 10.43 MiB/s, done.
Resolving deltas: 100% (57/57), done.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a client-grouped split because multiple content items can belong to the same client. Keeping each client entirely in either the training or test set prevents client-specific patterns from leaking across the split and gives a more honest estimate of performance on unseen clients.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

df = pd.read_csv(
    "/content/flyrank_ml_internship/data/raw/content_refresh_anonymized.csv"
)

df["is_declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X = df[features].fillna(0)
y = df["is_declining_label"]
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train:", X_train.shape)
print("Test:", X_test.shape)


Train: (23837, 14)
Test: (6163, 14)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I trained Logistic Regression and compared it with the Week-4 baseline using the same data, the same client-grouped split, and the same Precision@50 metric. This makes the comparison fair and shows whether the model improves the baseline for the top-priority content.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd

model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=5000))
])

model.fit(X_train, y_train)

scores = model.predict_proba(X_test)[:, 1]

top20_idx = scores.argsort()[-20:][::-1]
y_top20 = y_test.iloc[top20_idx]

model_precision_at_20 = y_top20.sum() / 20

print("Model Precision@20:", model_precision_at_20)

Model Precision@20: 0.75


In [10]:
baseline_precision_at_20 = 0.50  # Week-4 baseline

comparison = pd.DataFrame({
    "Method": ["Week-4 Baseline", "Logistic Regression"],
    "Precision@20": [baseline_precision_at_20, model_precision_at_20]
})

comparison

,Method,Precision@20
0,Week-4 Baseline,0.50
1,Logistic Regression,0.75


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Errors and interpretation

The model can be wrong when content has unusual traffic patterns or when recent changes in impressions and clicks do not clearly indicate a decline. The model mainly relies on observable engagement and traffic signals to identify content likely to decline. These predictions should be treated as decision-support rather than definitive judgments.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Error analysis
predicted = (scores >= 0.5).astype(int)

errors = pd.DataFrame({
    "actual": y_test.values,
    "predicted": predicted,
    "score": scores
})

false_positives = errors[
    (errors["actual"] == 0) & (errors["predicted"] == 1)
]

false_negatives = errors[
    (errors["actual"] == 1) & (errors["predicted"] == 0)
]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

False positives: 1570
False negatives: 1279


The model produced 1,570 false positives and 1,279 false negatives on the test set. This shows that the model still makes both types of errors when identifying declining content. The model relies on the available traffic, engagement, freshness, and search-performance features. The results are directional and should be used as decision-support rather than definitive judgments.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.